In [7]:
import pandas as pd
import numpy as np
from scipy import stats
import json
import os
import glob
import re
from pathlib import Path
from datetime import datetime

def read_sea_level_data(filename):
    """Read the sea level data from a text file"""
    data_rows = []
    
    with open(filename, 'r') as file:
        lines = file.readlines()
    
    # Find where the data starts
    data_start = None
    for i, line in enumerate(lines):
        if re.match(r'^\s*\d{1,2}\s+\d{4}', line.strip()):
            data_start = i
            break
    
    if data_start is None:
        raise ValueError("Could not find data start in file")
    
    # Parse each data line
    for line in lines[data_start:]:
        line = line.strip()
        
        if not line or line.startswith('Totals') or line.startswith('Mean sea level'):
            continue
            
        if not re.match(r'^\s*\d', line):
            continue
            
        parts = line.split()
        
        if len(parts) < 7:
            continue
            
        try:
            month = int(parts[0])
            year = int(parts[1])
            gaps = int(parts[2])
            good = int(parts[3])
            
            if len(parts) >= 7:
                minimum = float(parts[4]) if parts[4] != '' else None
                maximum = float(parts[5]) if parts[5] != '' else None
                mean = float(parts[6]) if parts[6] != '' else None
                std_dev = float(parts[7]) if len(parts) > 7 and parts[7] != '' else None
            else:
                minimum = maximum = mean = std_dev = None
            
            # Only add if we have valid mean data
            if mean is not None:
                data_rows.append({
                    'month': month,
                    'year': year,
                    'gaps': gaps,
                    'good': good,
                    'minimum': minimum,
                    'maximum': maximum,
                    'mean': mean,
                    'std_dev': std_dev,
                    'date': f"{year}-{month:02d}-01"
                })
            
        except (ValueError, IndexError) as e:
            continue
    
    return data_rows

def calculate_moving_average(data, window=12):
    """Calculate moving average for time series data"""
    if len(data) < window:
        return [None] * len(data)
    
    moving_averages = []
    for i in range(len(data)):
        if i < window // 2 or i >= len(data) - window // 2:
            moving_averages.append(None)
        else:
            start_idx = i - window // 2
            end_idx = start_idx + window
            window_data = [d['mean'] for d in data[start_idx:end_idx] if d['mean'] is not None]
            if len(window_data) >= window // 2:  # At least half the window has data
                moving_averages.append(sum(window_data) / len(window_data))
            else:
                moving_averages.append(None)
    
    return moving_averages

def calculate_linear_regression(data):
    """Calculate linear regression for sea level rise rate"""
    # Filter out None values
    valid_data = [(i, d['mean']) for i, d in enumerate(data) if d['mean'] is not None]
    
    if len(valid_data) < 24:  # Need at least 2 years of data
        return None
    
    x_values = [item[0] for item in valid_data]
    y_values = [item[1] for item in valid_data]
    
    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_values, y_values)
    
    # Convert slope from meters/month to mm/year
    rise_rate_mm_per_year = slope * 12 * 1000
    
    # Calculate confidence interval (95%)
    confidence_interval = 1.96 * std_err * 12 * 1000
    
    # Calculate trend line points
    trend_line = []
    for i, d in enumerate(data):
        if d['mean'] is not None:
            trend_value = intercept + slope * i
            trend_line.append(trend_value)
        else:
            trend_line.append(None)
    
    return {
        'rise_rate_mm_per_year': rise_rate_mm_per_year,
        'confidence_interval': confidence_interval,
        'r_squared': r_value ** 2,
        'p_value': p_value,
        'trend_line': trend_line,
        'slope': slope,
        'intercept': intercept
    }

def process_all_islands(folder_path):
    """Process all txt files and return structured data"""
    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))
    
    if not txt_files:
        raise ValueError(f"No .txt files found in {folder_path}")
    
    island_data = {}
    
    for file_path in txt_files:
        island_name = Path(file_path).stem
        island_name = island_name.replace('_sea_lvl', '').replace('_', ' ').title()
        
        try:
            data = read_sea_level_data(file_path)
            if len(data) > 0:
                island_data[island_name] = data
                print(f"✓ Processed {island_name}: {len(data)} data points")
            else:
                print(f"✗ No valid data found for {island_name}")
        except Exception as e:
            print(f"✗ Error processing {island_name}: {e}")
    
    return island_data

def prepare_chart_data(island_data):
    """Prepare data specifically for the two charts"""
    
    # 1. Rise Rates Data (for bar chart)
    rise_rates_data = []
    
    for island_name, data in island_data.items():
        regression_result = calculate_linear_regression(data)
        
        if regression_result:
            rise_rates_data.append({
                'island': island_name,
                'rate': float(round(regression_result['rise_rate_mm_per_year'], 2)),
                'confidence_interval': float(round(regression_result['confidence_interval'], 2)),
                'r_squared': float(round(regression_result['r_squared'], 4)),
                'p_value': float(regression_result['p_value']),
                'significant': bool(regression_result['p_value'] < 0.05),
                'data_points': int(len([d for d in data if d['mean'] is not None])),
                'start_year': int(min(d['year'] for d in data)),
                'end_year': int(max(d['year'] for d in data))
            })
    
    # Calculate regional average
    if rise_rates_data:
        individual_rates = [item['rate'] for item in rise_rates_data]
        regional_avg = sum(individual_rates) / len(individual_rates)
        regional_error = float(np.sqrt(sum([item['confidence_interval']**2 for item in rise_rates_data])) / len(rise_rates_data))
        
        rise_rates_data.append({
            'island': 'Regional Average',
            'rate': float(round(regional_avg, 2)),
            'confidence_interval': float(round(regional_error, 2)),
            'r_squared': None,
            'p_value': None,
            'significant': None,
            'data_points': int(sum(item['data_points'] for item in rise_rates_data)),
            'start_year': int(min(item['start_year'] for item in rise_rates_data)),
            'end_year': int(max(item['end_year'] for item in rise_rates_data)),
            'is_average': True
        })
    
    # 2. Trends Data (for line chart with shaded areas)
    trends_data = []
    
    # Find common time range
    all_dates = set()
    for data in island_data.values():
        for d in data:
            all_dates.add(d['date'])
    
    sorted_dates = sorted(list(all_dates))
    
    # Create trend data for each date
    for date in sorted_dates:
        date_entry = {'date': date}
        
        for island_name, data in island_data.items():
            # Find data for this date
            date_data = next((d for d in data if d['date'] == date), None)
            
            if date_data and date_data['mean'] is not None:
                # Add raw data
                date_entry[f"{island_name}_raw"] = float(round(date_data['mean'], 4))
                if date_data['minimum'] is not None:
                    date_entry[f"{island_name}_min"] = float(date_data['minimum'])
                if date_data['maximum'] is not None:
                    date_entry[f"{island_name}_max"] = float(date_data['maximum'])
            
            # Calculate moving average for this point
            date_index = next((i for i, d in enumerate(data) if d['date'] == date), None)
            if date_index is not None:
                moving_avg = calculate_moving_average(data)
                if moving_avg[date_index] is not None:
                    date_entry[f"{island_name}_ma"] = float(round(moving_avg[date_index], 4))
            
            # Add trend line value
            regression_result = calculate_linear_regression(data)
            if regression_result and date_index is not None:
                trend_value = regression_result['trend_line'][date_index]
                if trend_value is not None:
                    date_entry[f"{island_name}_trend"] = float(round(trend_value, 4))
        
        trends_data.append(date_entry)
    
    return rise_rates_data, trends_data

def save_data_for_react(island_data, rise_rates_data, trends_data, output_dir="react_data"):
    """Save processed data in JSON format for React app"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Save rise rates data (for bar chart)
    rise_rates_file = os.path.join(output_dir, "rise_rates_data.json")
    with open(rise_rates_file, 'w') as f:
        json.dump(rise_rates_data, f, indent=2)
    print(f"✓ Rise rates data saved to: {rise_rates_file}")
    
    # 2. Save trends data (for line chart)
    trends_file = os.path.join(output_dir, "trends_data.json")
    with open(trends_file, 'w') as f:
        json.dump(trends_data, f, indent=2)
    print(f"✓ Trends data saved to: {trends_file}")
    
    # 3. Save raw island data (for reference)
    raw_data_file = os.path.join(output_dir, "raw_island_data.json")
    with open(raw_data_file, 'w') as f:
        json.dump(island_data, f, indent=2)
    print(f"✓ Raw island data saved to: {raw_data_file}")
    
    # 4. Save metadata
    metadata = {
        'export_date': datetime.now().isoformat(),
        'description': 'Pacific Islands Sea Level Rise Analysis Data',
        'islands_processed': list(island_data.keys()),
        'total_data_points': sum(len(data) for data in island_data.values()),
        'date_range': {
            'start': min(min(d['date'] for d in data) for data in island_data.values()),
            'end': max(max(d['date'] for d in data) for data in island_data.values())
        },
        'regional_average_rate': float(next((item['rate'] for item in rise_rates_data if item.get('is_average')), 0)),
        'rate_range': {
            'min': float(min(item['rate'] for item in rise_rates_data if not item.get('is_average'))),
            'max': float(max(item['rate'] for item in rise_rates_data if not item.get('is_average')))
        }
    }
    
    metadata_file = os.path.join(output_dir, "metadata.json")
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved to: {metadata_file}")
    
    # 5. Create summary CSV for easy viewing
    rise_rates_df = pd.DataFrame(rise_rates_data)
    csv_file = os.path.join(output_dir, "rise_rates_summary.csv")
    rise_rates_df.to_csv(csv_file, index=False)
    print(f"✓ Rise rates summary CSV saved to: {csv_file}")
    
    return {
        'rise_rates_file': rise_rates_file,
        'trends_file': trends_file,
        'raw_data_file': raw_data_file,
        'metadata_file': metadata_file,
        'csv_file': csv_file
    }

def print_data_summary(rise_rates_data, metadata):
    """Print a summary of the processed data"""
    print("\n" + "="*60)
    print("         SEA LEVEL DATA PROCESSING SUMMARY")
    print("="*60)
    
    print(f"\nDATA OVERVIEW:")
    print(f"• Islands processed: {len(metadata['islands_processed'])}")
    print(f"• Total data points: {metadata['total_data_points']:,}")
    print(f"• Date range: {metadata['date_range']['start']} to {metadata['date_range']['end']}")
    print(f"• Export date: {metadata['export_date']}")
    
    print(f"\nRISE RATES SUMMARY:")
    individual_rates = [item for item in rise_rates_data if not item.get('is_average')]
    print(f"• Regional average: {metadata['regional_average_rate']:.2f} mm/year")
    print(f"• Rate range: {metadata['rate_range']['min']:.2f} to {metadata['rate_range']['max']:.2f} mm/year")
    print(f"• Significant trends: {sum(1 for item in individual_rates if item['significant'])}/{len(individual_rates)} islands")
    
    print(f"\nINDIVIDUAL ISLAND RATES:")
    for item in sorted(individual_rates, key=lambda x: x['rate'], reverse=True):
        significance = "***" if item['p_value'] < 0.001 else \
                     "**" if item['p_value'] < 0.01 else \
                     "*" if item['p_value'] < 0.05 else ""
        print(f"• {item['island']:20s}: {item['rate']:6.2f} ± {item['confidence_interval']:.2f} mm/year {significance}")
    
    print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05")

def main():
    """Main function to process data and save for React"""
    
    folder_path = "/Users/alexfion/Dropbox/Alex/dataViz/AShymanskaya.github.io/portfolio/visualizations/pacific_dataviz/sea_lvl_rise"  # Change this to your data folder
    
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
        print("Please make sure the folder exists and contains .txt files with sea level data.")
        return
    
    try:
        print("Step 1: Processing sea level data files...")
        island_data = process_all_islands(folder_path)
        
        if not island_data:
            print("No valid data files found.")
            return
        
        print(f"\nStep 2: Calculating trends and preparing chart data...")
        rise_rates_data, trends_data = prepare_chart_data(island_data)
        
        print(f"\nStep 3: Saving data for React application...")
        file_paths = save_data_for_react(island_data, rise_rates_data, trends_data)
        
        # Load metadata for summary
        with open(file_paths['metadata_file'], 'r') as f:
            metadata = json.load(f)
        
        print_data_summary(rise_rates_data, metadata)
        
        print(f"\n🎉 SUCCESS! All data has been processed and saved.")
        print(f"📁 Files saved in 'react_data/' directory:")
        for file_type, file_path in file_paths.items():
            print(f"   • {file_type}: {file_path}")
        
        print(f"\n💡 Next steps:")
        print(f"   1. Import the JSON files into your React app")
        print(f"   2. Use 'rise_rates_data.json' for the bar chart")
        print(f"   3. Use 'trends_data.json' for the line chart with shaded areas")
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Step 1: Processing sea level data files...
✓ Processed Samoa: 386 data points
✓ Processed Kiribati: 382 data points
✓ Processed Fiji: 390 data points
✓ Processed Nauru: 364 data points
✓ Processed Solomon: 364 data points
✓ Processed Niue: 71 data points
✓ Processed Tonga: 388 data points
✓ Processed Tuvalu: 381 data points
✓ Processed Papua: 346 data points
✓ Processed Marshall: 377 data points
✓ Processed Cook: 384 data points
✓ Processed Micronesia: 262 data points
✓ Processed Vanuatu: 378 data points

Step 2: Calculating trends and preparing chart data...

Step 3: Saving data for React application...
✓ Rise rates data saved to: react_data/rise_rates_data.json
✓ Trends data saved to: react_data/trends_data.json
✓ Raw island data saved to: react_data/raw_island_data.json
✓ Metadata saved to: react_data/metadata.json
✓ Rise rates summary CSV saved to: react_data/rise_rates_summary.csv

         SEA LEVEL DATA PROCESSING SUMMARY

DATA OVERVIEW:
• Islands processed: 13
• Total data poin